In [18]:
import os
import pandas as pd
import numpy as np
import networkx as nx 
import matplotlib.pyplot as plt
from community import community_louvain 
from networkx.algorithms.community.quality import modularity, performance, coverage
from networkx.algorithms.cuts import conductance
from sklearn.metrics import adjusted_rand_score, rand_score, normalized_mutual_info_score
from scipy.stats import spearmanr
import torch
from collections import Counter
from scipy import stats


In [ ]:
def read_data(num_nodes=-1):

    """ Reads data of politician, their party and who they follow into df_data """

    members = pd.read_csv('raw_data/raw_data/all_congress_members.csv')
    members['current_twitter_name'] = members['new_twitter_name'].fillna(members['twitter_name'])

    df_data = pd.DataFrame()
    df_data[['current_twitter_name','party']] = members[['current_twitter_name','party']].copy()
    if num_nodes != -1:
        df_data = df_data.head(num_nodes)

    following_data = []
    
    for twitter_name in df_data['current_twitter_name']:
        filename = f"twitterExport_{twitter_name}_Following.csv"
        filepath = os.path.join('raw_data/raw_data/following_data_raw', filename)

        if os.path.exists(filepath):
            followed_users = pd.read_csv(filepath)['user_name'].tolist()
        else:
            followed_users = []
        
        following_data.append(followed_users)

    df_data['complete_following'] = following_data
    df_data = df_data[df_data['complete_following'].apply(lambda x: x != [])]

    # Convert the list of all current twitter names to a set for faster lookup
    current_names_set = set(df_data['current_twitter_name'].values)

    # Filter out following names that also appear in df_data['current_twitter_name']
    filtered_following_data = []
    for follow_list in df_data['complete_following']:
        filtered_list = [name for name in follow_list if name not in current_names_set]
        filtered_following_data.append(filtered_list)

    df_data['following'] = filtered_following_data
    df_data = df_data[df_data['following'].apply(lambda x: x != [])]

    republican_df = df_data[df_data['party'] == "R"]
    democratic_df = df_data[df_data['party'] == "D"]

    return df_data, republican_df, democratic_df

In [20]:
def build_network(df_data, k, hits, weighted):

    """ Builds network of politicians, with edges between them if they share k followers """

    G = nx.Graph()
    
    # Add nodes
    names = df_data['current_twitter_name'].tolist()

    for _, row in df_data.iterrows():
        politician = row['current_twitter_name']        
        G.add_node(politician, party=row['party'])
    
    # Add edges based on shared following
    for i in range(len(df_data)):
        for j in range(i + 1, len(df_data)):
            # Get the sets of following of each politcian
            # Uses the "important_following" column if we incorporate hits, otherwise defaults to the "following" column
            if hits:
                following_i = set(df_data.iloc[i]['important_following'])
                following_j = set(df_data.iloc[j]['important_following'])
            else:
                following_i = set(df_data.iloc[i]['following'])
                following_j = set(df_data.iloc[j]['following'])
            
            # Get the shared following
            shared_following = following_i.intersection(following_j)
            if len(shared_following) >= k:
                if weighted == False:
                    G.add_edge(names[i], names[j])
                else:
                    G.add_edge(names[i], names[j], weight=len(shared_following))
    
    return G

In [21]:
def classify_communities(alignment):
    ratio_list = []
    community_label = {}
    for community_id, parties_total in alignment.items():
        ratio = parties_total['R']/(parties_total['R']+parties_total['D'])
        ratio_list.append(ratio)

    if ratio_list[0]>ratio_list[1]:
        Republican = 1
        Democrat = 2
    else:
        Republican = 2
        Democrat = 1

    community_label[Republican] = 'Republican'
    community_label[Democrat] = 'Democrat' 
    # print(ratio_list, community_label)
    return community_label
    

In [22]:
def merge_communities_by_edge_count(G, communities):
    """Merge a list of communities into exactly two based on the edge counts between them.

    Args:
        G (networkx.Graph): The graph from which the communities are derived.
        communities (list of set): List of communities to merge, where each community is a set of nodes.

    Returns:
        list of set: Two communities merged based on edge connectivity.
    """
    if len(communities) < 2:
        raise ValueError("There must be at least two communities to merge into two.")
    
    # Convert frozensets or other iterables to sets if necessary
    communities = [set(community) for community in communities]
    
    # Sort communities by size in descending order
    communities = sorted(communities, key=len, reverse=True)
    
    # Start with the two largest communities
    large_community_one, large_community_two = communities[:2]
    
    # Function to count edges between two sets of nodes
    def count_edges_between(set1, set2):
        return sum(1 for node in set1 if set(G.neighbors(node)) & set2)

    # Merge remaining communities into one of these two
    for community in communities[2:]:
        edges_to_one = count_edges_between(community, large_community_one)
        edges_to_two = count_edges_between(community, large_community_two)
        
        if edges_to_one > edges_to_two:
            large_community_one.update(community)
        else:
            large_community_two.update(community)

    return [large_community_one, large_community_two]

In [23]:
from networkx.algorithms.community import greedy_modularity_communities, modularity

def apply_greedy_modularity_two_partitions(G):
    """Applies greedy modularity community detection and merges results into two communities.

    Args:
        G (networkx.Graph): The graph on which to apply the community detection.

    Returns:
        tuple: Contains the two communities, party alignment, modularity of the partition, and community labels.
    """
    # Detect communities using greedy modularity maximization
    initial_communities = greedy_modularity_communities(G)
    # print(len(initial_communities))

    # Merge communities into two
    if len(initial_communities) > 2:
        communities = merge_communities_by_edge_count(G, initial_communities)
        # communities = merge_communities_max_modularity(G, initial_communities)
        # communities = merge_into_two_communities(initial_communities)
    else:
        communities = [set(community) for community in initial_communities]

    # Analyze alignment with political parties
    alignment = {1: {'R': 0, 'D': 0}, 2: {'R': 0, 'D': 0}}
    for idx, community in enumerate(communities):
        for node in community:
            party = G.nodes[node]['party']
            alignment[idx + 1][party] += 1

    # Calculate modularity for the two communities
    if G.number_of_edges() != 0:
        mod_value = modularity(G, communities)
    # print(f"Modularity of the partition: {mod_value}")
    else:
        mod_value = 0

    # Classify communities based on dominant party
    comm_labels = classify_communities(alignment)

    return communities, alignment, mod_value, comm_labels


In [24]:
def build_bipartite_network(df_data):

    """ Builds directed network using politicians and their followings as nodes
    and directed edges based on if one node follows the other. Used for finding importance
    of websites to filter network """

    G = nx.DiGraph()

    # Add edges based on the politicians and the people they follow
    for _, row in df_data.iterrows():
        politician = row['current_twitter_name']
        following_list = row['following']  # List of people the politician follows
        
        # Add the politician as a node
        G.add_node(politician, party=row['party'])
        
        # Add edges for each person the politician follows
        for followee in following_list:
            G.add_node(followee)  # Add the followed account as a node
            G.add_edge(politician, followee)  # Directed edge from politician to followed account
    
    return G

In [25]:
def calc_hits_values(G, df_data):
    # Compute HITS scores
    hub_scores, authority_scores = nx.hits(G, max_iter=100, tol=1e-7)
    
    # Store the HITS scores as node attributes
    for node in G.nodes():
        if node not in df_data['current_twitter_name'].values:
            G.nodes[node]['hub_score'] = hub_scores.get(node, 0)
            G.nodes[node]['authority_score'] = authority_scores.get(node, 0)
        else:
            authority_scores.pop(node)
            hub_scores.pop(node)
    
    # returning below to easily calc percentiles etc
    return authority_scores, hub_scores

In [26]:
def calc_centrality_values(G, df_data):
    
    """ Betweenness Centrality: How often a node acts as a bridge along the shortest path between two other nodes - high means node has considerable influence over spread of info through the network
        Closeness Centrality: Measures how close a node is to all other node in the network, based on shortest path - high means node can interact quickly with others in the network
        YES = Eigenvector Centrality: Measures connection quality - connection to high scoring nodes contribute more to the score than low scoring nodes
        YES = Degree Centrality: Measures the number of direct connections to other nodes - high means more connections """
    
    eigenvector_centrality, degree_centrality = nx.eigenvector_centrality(G), nx.degree_centrality(G)
    # Store the scores as node attributes
    for node in G.nodes():
        if node not in df_data['current_twitter_name'].values:
            G.nodes[node]['eigenvector'] = eigenvector_centrality.get(node, 0)
            G.nodes[node]['degree'] = degree_centrality.get(node, 0)
        else:
            eigenvector_centrality.pop(node)
            degree_centrality.pop(node)

    return eigenvector_centrality, degree_centrality

In [27]:
def add_important_following_column(G, df_data, auth_thresh):

    """ Use calculated HITS values to add new column of the following if 
    they are above the authority value threshold """

    # Create a set of nodes to retain
    retained_nodes = set()
    for node in G.nodes:
        if G.nodes[node]['authority_score'] >= auth_thresh or (
            node in df_data['current_twitter_name'].values):
            retained_nodes.add(node)

    # Create a new column 'important_following'
    important_following = []
    for _, row in df_data.iterrows():
        following_list = row['following']
        
        # Filter the followings to retain only those in the retained_nodes set
        filtered_following = [f for f in following_list if f in retained_nodes]
        important_following.append(filtered_following)

    # Add the new column to the dataframe
    df_data['important_following'] = important_following
    return df_data

In [28]:
df_data, republican_df, democratic_df = read_data(-1)

In [29]:
def calculate_metrics(alignment, community_label):
    """
    Calculate FFC, ARI, RI, and NMI from alignment data and graph nodes.

    Parameters:
        alignment (dict): Alignment data of communities to political parties.
        node_to_community (dict): Mapping of nodes to their assigned community.
        G (Graph): The graph with nodes and party information.

    Returns:
        dict: Calculated metrics.
    """
    # print(community_label[1])
    if community_label[1] == 'Republican':
        Republican = 1
        Democrat = 2
    else:
        Republican = 2
        Democrat = 1

    # print(Republican, Democrat)
    # Extract counts
    true_positive_R = alignment[Republican]["R"]
    false_positive_R = alignment[Republican]["D"]
    true_positive_D = alignment[Democrat]["D"]
    false_positive_D = alignment[Democrat]["R"]
    # Precision and Recall for R
    precision_R = true_positive_R / (true_positive_R + false_positive_R) if (true_positive_R + false_positive_R) > 0 else 0
    recall_R = true_positive_R / (true_positive_R + false_positive_D) if (true_positive_R + false_positive_D) > 0 else 0
    
    # Precision and Recall for D
    precision_D = true_positive_D / (true_positive_D + false_positive_D) if (true_positive_D + false_positive_D) > 0 else 0
    recall_D = true_positive_D / (true_positive_D + false_positive_R) if (true_positive_D + false_positive_R) > 0 else 0

    # Macro Average (Optional)
    precision_macro = (precision_R + precision_D) / 2
    recall_macro = (recall_R + recall_D) / 2

    f1_score = 2*precision_macro*recall_macro/(precision_macro+recall_macro)  if (precision_macro+recall_macro) > 0 else 0
    accuracy = (true_positive_R+true_positive_D)/(true_positive_D+true_positive_R+false_positive_R+false_positive_D)


    return {
        "accuracy": accuracy,
        "f1": f1_score
    }


In [30]:
def flatten(list_of_lists):
    """ Flatten a list of lists to a single list. """
    return [item for sublist in list_of_lists for item in sublist]


def evaluate_metrics(df_data, threshold_value, metric_percentiles):
    """
    Analyze the effect of network metric thresholds and percentile filters on alignment metrics.

    Parameters:
        df_data (DataFrame): Data for the network.
        threshold_value (int): The threshold value for filtering network connections.
        metric_percentiles (dict): Dictionary with metrics as keys and percentiles as values,
                                   specifying the filtering threshold for each metric.

    Returns:
        dict: Dictionary containing the results with accuracy, f1, and modularity for each scenario.
    """
    # metrics = ['authority', 'eigenvector', 'degree']
    # results = {metric: {'accuracy': [], 'f1': [], 'modularity': []} for metric in metrics}
    results = {'accuracy': 0, 'f1': 0, 'modularity': 0}  # For the scenario without any metrics
    
    # Assume these functions are defined elsewhere in your codebase
    G_directed = build_bipartite_network(df_data)
    authority_scores, _ = calc_hits_values(G_directed, df_data)
    # eigenvector_centrality, degree_centrality = calc_centrality_values(G_directed, df_data)
    degree_centrality = {node: G_directed.degree(node) for node in G_directed.nodes()}

    # Mapping metric names to their corresponding data
    metric_data = {
        'authority': authority_scores,
        # 'eigenvector': eigenvector_centrality,
        'degree': degree_centrality
    }

    important_followees = set(flatten(df_data['following'].tolist()))  # Initialize with all followees
    for metric, scores in metric_data.items():
        if metric_percentiles[metric] == 0:
            filtered_followees = set(scores.keys())
            print(len(filtered_followees))
        else:
            percentile_threshold = np.percentile(list(scores.values()), metric_percentiles[metric])
            filtered_followees = {node for node, score in scores.items() if score >= percentile_threshold}
        important_followees.intersection_update(filtered_followees)  # Apply combined filtering

    df_data['important_following'] = df_data['following'].apply(
        lambda followings: [f for f in followings if f in important_followees]
    )

    # Analyze network with the specific threshold
    G = build_network(df_data, threshold_value, True, True)  # Adjust this as necessary
    try:
        _, alignment, mod_value, comm_labels = apply_greedy_modularity_two_partitions(G)
        a_f1_metrics = calculate_metrics(alignment, comm_labels)

    except:
        a_f1_metrics = {}
        a_f1_metrics['accuracy'] = 0
        a_f1_metrics['f1'] = 0
        mod_value = 0

    # Store results for the specified threshold and metrics
    results['accuracy'] = a_f1_metrics['accuracy']
    results['f1'] = a_f1_metrics['f1']
    results['modularity'] = mod_value

    return results


In [ ]:
from deap import base, creator, tools, algorithms
import random

# Create fitness and individual classes
creator.create("FitnessMax", base.Fitness, weights=(1.0, 1.0, 1.0))  # We're maximizing
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_threshold", random.randint, 1, 20)
toolbox.register("attr_percentile", random.randint, 0, 100)

# Initialize individual and population
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_threshold, toolbox.attr_percentile, toolbox.attr_percentile), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


c:\Users\tusha\anaconda3\lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
c:\Users\tusha\anaconda3\lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [ ]:
def evaluate(individual):
    threshold_value, authority_percentile, degree_percentile = individual
    metric_percentiles = {
        'authority': authority_percentile,
        'degree': degree_percentile
    }
    results = evaluate_metrics(df_data, threshold_value, metric_percentiles)
    print(results)
    # Assuming results return a dict with 'accuracy', 'f1', 'modularity'
    return results['accuracy'], results['f1'], results['modularity']

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=[1,0,0], up=[20,100,100], indpb=0.2)
toolbox.register("select", tools.selNSGA2)


In [33]:
def best_metrics_finder():
    population = toolbox.population(n=100)
    NGEN = 50
    CXPB, MUTPB = 0.5, 0.2  # Crossover and mutation probabilities

    for gen in range(NGEN):
        # Select the next generation individuals
        offspring = toolbox.select(population, len(population))
        # Clone the selected individuals
        offspring = list(map(toolbox.clone, offspring))

        # Apply crossover and mutation on the offspring
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Replace the old population by the offspring
        population[:] = offspring

    return population

final_pop = best_metrics_finder()
top_individuals = tools.sortNondominated(final_pop, len(final_pop), first_front_only=True)
for ind in top_individuals[0]:
    print("Individual:", ind, "Fitness:", ind.fitness.values)


c:\Users\tusha\anaconda3\lib\site-packages\networkx\algorithms\link_analysis\hits_alg.py:78: FutureWarning: adjacency_matrix will return a scipy.sparse array instead of a matrix in Networkx 3.0.
  A = nx.adjacency_matrix(G, nodelist=list(G), dtype=float)


{'accuracy': 0.9451219512195121, 'f1': 0.9456650864152167, 'modularity': 0.14480903250905852}


c:\Users\tusha\anaconda3\lib\site-packages\networkx\algorithms\link_analysis\hits_alg.py:78: FutureWarning: adjacency_matrix will return a scipy.sparse array instead of a matrix in Networkx 3.0.
  A = nx.adjacency_matrix(G, nodelist=list(G), dtype=float)


{'accuracy': 0.8841463414634146, 'f1': 0.8834695796639135, 'modularity': 0.0976818710161334}
{'accuracy': 0.8841463414634146, 'f1': 0.8834695796639135, 'modularity': 0.10062784371839695}
{'accuracy': 0.9634146341463414, 'f1': 0.9637278181991955, 'modularity': 0.15349622455816866}
{'accuracy': 0.9451219512195121, 'f1': 0.9456650864152167, 'modularity': 0.14468060732385057}
{'accuracy': 0.9451219512195121, 'f1': 0.9466141335918843, 'modularity': 0.14019861656978683}
{'accuracy': 0.9573170731707317, 'f1': 0.9588869390824769, 'modularity': 0.1487381920147902}
{'accuracy': 0, 'f1': 0, 'modularity': 0}
{'accuracy': 0.9451219512195121, 'f1': 0.9456650864152167, 'modularity': 0.1518445066843884}
{'accuracy': 0.9573170731707317, 'f1': 0.9579488908325434, 'modularity': 0.1547742838576692}
{'accuracy': 0.9573170731707317, 'f1': 0.9579488908325434, 'modularity': 0.15435586438670265}
{'accuracy': 0.9512195121951219, 'f1': 0.9514357489629597, 'modularity': 0.1501161860853637}
{'accuracy': 0.94512195

In [34]:
for ind in top_individuals[0]:
    print("Individual:", ind, "Fitness:", ind.fitness.values)


Individual: [14, 0, 74] Fitness: (0.9634146341463414, 0.9637278181991955, 0.15491918214118167)
Individual: [5, 11, 89] Fitness: (0.9512195121951219, 0.9522452579456108, 0.16637253184574174)
Individual: [20, 21, 79] Fitness: (0.9634146341463414, 0.9629685383110042, 0.16170547914313205)
